# Behavior cloning — imitate an expert

**Agents & RL** · the simplest imitation-learning recipe (how many robot/driving policies start).

We collect (state → expert action) demonstrations and train a network to copy them — pure supervised learning — then roll out the learned policy and measure success.

> CPU is fine.

In [ ]:
import os, torch, torch.nn as nn, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 800)); N = 6; GOAL = (N - 1, N - 1)
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]
def feat(p): v = torch.zeros(2 * N); v[p[0]] = 1.0; v[N + p[1]] = 1.0; return v
def expert(p):                                   # greedy: reduce Manhattan distance to goal
    if p[0] < GOAL[0]: return 1
    if p[1] < GOAL[1]: return 3
    return 0

## 1 · Collect demonstrations from the expert

In [ ]:
X, Y = [], []
for _ in range(2000):
    p = (torch.randint(0, N, (1,)).item(), torch.randint(0, N, (1,)).item())
    if p == GOAL: continue
    X.append(feat(p)); Y.append(expert(p))
X = torch.stack(X).to(device); Y = torch.tensor(Y, device=device)
print("demonstrations:", X.shape[0])

## 2 · Train the policy to copy the expert

In [ ]:
policy = nn.Sequential(nn.Linear(2 * N, 64), nn.ReLU(), nn.Linear(64, 4)).to(device)
opt = torch.optim.Adam(policy.parameters(), 3e-3); hist = []
for step in range(STEPS + 1):
    loss = nn.functional.cross_entropy(policy(X), Y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % max(1, STEPS // 10) == 0:
        acc = (policy(X).argmax(1) == Y).float().mean().item(); hist.append((step, round(acc, 3))); print(f"step {step:4d}  imitation acc {acc:.3f}")

## 3 · Roll out the learned policy — does it reach the goal?

In [ ]:
@torch.no_grad()
def rollout(p):
    for _ in range(2 * N):
        if p == GOAL: return True
        a = policy(feat(p).to(device)).argmax().item()
        p = (min(max(p[0] + MOVES[a][0], 0), N - 1), min(max(p[1] + MOVES[a][1], 0), N - 1))
    return p == GOAL
succ = sum(rollout((i // N, i % N)) for i in range(N * N) if (i // N, i % N) != GOAL) / (N * N - 1)
print(f"success rate from every start cell: {succ:.3f}")

## 4 · Compare — imitation accuracy over training

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(*zip(*hist), "-o"); ax.set_xlabel("step"); ax.set_ylabel("imitation accuracy"); ax.grid(alpha=.3); ax.set_title(f"behavior cloning — rollout success {succ:.0%}")
plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/AG_behavior_cloning/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/AG_behavior_cloning"; os.makedirs(run, exist_ok=True)
torch.save(policy.state_dict(), f"{run}/policy.pt")
json.dump({"imitation_acc": hist, "rollout_success": succ}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## How this links to tracks A–D
Imitation learns **A** human-like skills and **D** embodied policies.

### Where to go next
- BC suffers from **distribution shift**; fix it with **DAgger** (query the expert on the policy's own states).
- Imitate from images with a CNN encoder; combine with RL fine-tuning (track AG).